## Import dependencies

In [19]:
import pandas as pd
import mlflow.sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from src.studentrisk_ai.data_utils import process_data

# Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier


## MLflow setup

In [20]:
from pathlib import Path

tracking_db = Path("../../mlflow.db").resolve()
mlflow.set_tracking_uri(f"sqlite:///{tracking_db}")
mlflow.set_experiment("Student Mental Health Risk Prediction")

X_train, y_train, X_test, y_test = process_data(
    "../../data/raw/student_mental_health.csv"
)

print(X_train.shape, X_test.shape)
target_encoder = LabelEncoder()
y_train_encoded = target_encoder.fit_transform(y_train)
y_test_encoded = target_encoder.transform(y_test)

(1600, 18) (400, 18)


## Train and compare models

In [21]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        eval_metric="mlogloss",
    ),
}

results = []
predictions = {}

for model_name, model in models.items():
    model_pipeline = Pipeline([
        ("model", model),
    ])
    with mlflow.start_run(run_name=model_name):
        model_pipeline.fit(X_train, y_train_encoded)
        y_pred_encoded = model_pipeline.predict(X_test)
        y_pred = target_encoder.inverse_transform(y_pred_encoded.astype(int))
        predictions[model_name] = y_pred

        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
            "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
            "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
        }
        mlflow.log_params(model_pipeline.get_params())
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(
            model_pipeline,
            name="model",
            skops_trusted_types=[
                "sklearn.tree._tree.Tree",
                "xgboost.core.Booster",
                "xgboost.sklearn.XGBClassifier",
            ],
        )

        results.append({"model": model_name, **metrics})

results_df = pd.DataFrame(results).set_index("model").round(3)
display(results_df)


2026/11/11 13:50:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/11 13:50:58 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/11/11 13:51:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


,accuracy,precision,recall,f1_score
model,,,,
Logistic Regression,0.758,0.755,0.758,0.756
Random Forest,0.715,0.712,0.715,0.713
XGBoost,0.718,0.714,0.718,0.716


## Reports and confusion matrices

In [22]:
for model_name, y_pred in predictions.items():
    print(f"\n{model_name}")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred, labels=["Low", "Medium", "High"]))



Logistic Regression
              precision    recall  f1-score   support

        High       0.80      0.84      0.82       136
         Low       0.82      0.82      0.82       132
      Medium       0.64      0.61      0.63       132

    accuracy                           0.76       400
   macro avg       0.75      0.76      0.76       400
weighted avg       0.76      0.76      0.76       400

Confusion matrix:
[[108  23   1]
 [ 24  81  27]
 [  0  22 114]]

Random Forest
              precision    recall  f1-score   support

        High       0.76      0.80      0.78       136
         Low       0.78      0.78      0.78       132
      Medium       0.60      0.56      0.58       132

    accuracy                           0.71       400
   macro avg       0.71      0.71      0.71       400
weighted avg       0.71      0.71      0.71       400

Confusion matrix:
[[103  25   4]
 [ 27  74  31]
 [  2  25 109]]

XGBoost
              precision    recall  f1-score   support

        Hi